In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Imports, rutas y parámetros

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/dataset_senales_acusticas")

METADATA_DIR = BASE_DIR / "metadata"
STATISTICS_DIR = BASE_DIR / "statistics"

EVENTS_BASE_PATH = METADATA_DIR / "events_manifest_base.csv"
RECORDINGS_BASE_PATH = METADATA_DIR / "recordings_manifest_base.csv"

MASTER_PATH = METADATA_DIR / "master_manifest.csv"
RECORDINGS_PATH = METADATA_DIR / "recordings_manifest.csv"
TARGET_SPECIES_PATH = METADATA_DIR / "target_species.csv"
DETECTOR_PATH = METADATA_DIR / "detector_manifest.csv"
CLASSIFIER_PATH = METADATA_DIR / "classifier_manifest.csv"
DATASET_SUMMARY_PATH = METADATA_DIR / "dataset_summary.csv"
MODELING_CONFIG_PATH = METADATA_DIR / "modeling_config.json"

SPECIES_AUDIT_PATH = STATISTICS_DIR / "species_support_audit.csv"
CLASS_DISTRIBUTION_PATH = (
    STATISTICS_DIR / "classifier_class_distribution.csv"
)
INTEGRITY_REPORT_PATH = STATISTICS_DIR / "integrity_report.csv"

MIN_EVENTS = 200
MIN_RECORDINGS = 15

COLOMBIAN_ELIGIBILITY_DATASETS = {
    "PteroSet",
    "WABAD-CO",
}

OTHER_LABEL = "OTHER_SPECIES"
OTHER_CLASS_CODE = 0

METADATA_DIR.mkdir(parents=True, exist_ok=True)
STATISTICS_DIR.mkdir(parents=True, exist_ok=True)

if not EVENTS_BASE_PATH.exists():
    raise FileNotFoundError(
        f"No existe {EVENTS_BASE_PATH}. Ejecuta primero el notebook 01."
    )

if not RECORDINGS_BASE_PATH.exists():
    raise FileNotFoundError(
        f"No existe {RECORDINGS_BASE_PATH}. Ejecuta primero el notebook 01."
    )

print("Proyecto:", BASE_DIR)
print("MIN_EVENTS:", MIN_EVENTS)
print("MIN_RECORDINGS:", MIN_RECORDINGS)
print(
    "Fuentes colombianas para elegibilidad:",
    sorted(COLOMBIAN_ELIGIBILITY_DATASETS),
)

# 1. Carga y validación de los manifests base

In [ ]:
events = pd.read_csv(EVENTS_BASE_PATH)
recordings_base = pd.read_csv(RECORDINGS_BASE_PATH)

print("Eventos base:", len(events))
print("Audios físicos base:", len(recordings_base))
print("Grabaciones anotadas:", events["recording_uid"].nunique())

required_event_columns = {
    "event_id",
    "dataset",
    "recording",
    "recording_key",
    "recording_uid",
    "source_group",
    "source_archive",
    "country",
    "species_scientific",
    "identified",
    "t_min",
    "t_max",
    "f_min_original",
    "f_max_original",
    "f_min",
    "f_max",
    "duration_event_sec",
    "bandwidth_hz",
    "use_detector",
    "audio_available",
}

required_recording_columns = {
    "recording_uid",
    "dataset",
    "recording_key",
    "source_recording",
    "source_group",
    "source_archive",
    "country",
    "status",
    "has_annotations",
    "normalized_path",
    "duration_sec",
    "normalized_sample_rate",
    "normalized_channels",
    "audio_available",
}

missing_events = required_event_columns - set(events.columns)
missing_recordings = required_recording_columns - set(recordings_base.columns)

if missing_events:
    raise KeyError(
        f"events_manifest_base.csv carece de: {sorted(missing_events)}"
    )

if missing_recordings:
    raise KeyError(
        f"recordings_manifest_base.csv carece de: {sorted(missing_recordings)}"
    )


def to_bool(series):
    if series.dtype == bool:
        return series

    converted = (
        series.astype(str)
        .str.strip()
        .str.casefold()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

    if converted.isna().any():
        bad = series.loc[converted.isna()].drop_duplicates().tolist()
        raise ValueError(
            f"No se pudo convertir a booleano. Valores: {bad[:20]}"
        )

    return converted.astype(bool)


for column in ["identified", "use_detector", "audio_available"]:
    events[column] = to_bool(events[column])

for column in ["has_annotations", "audio_available"]:
    recordings_base[column] = to_bool(
        recordings_base[column]
    )

events["species_scientific"] = (
    events["species_scientific"].astype("string")
)

assert len(events) == 48465
assert events["event_id"].is_unique
assert events["recording_uid"].nunique() == 1011

assert len(recordings_base) == 1017
assert recordings_base["recording_uid"].is_unique
assert int(recordings_base["has_annotations"].sum()) == 1011

print("Validación base correcta.")

# 2. Qué cuenta como especie taxonómicamente exacta

Para el clasificador solo aceptamos etiquetas que representen una especie científica binomial completa, por ejemplo:

```text
tyrannus melancholicus
crypturellus soui
```

Se excluyen etiquetas incompletas o corruptas como:

```text
psittacidae sp.
picidae
tyrannidae sp 1
â€“
```

La regla se audita explícitamente; no se eliminan las filas del `master_manifest`.

In [ ]:
EXACT_SPECIES_PATTERN = re.compile(
    r"^[a-z][a-z-]+ [a-z][a-z-]+$"
)


def is_exact_species_name(value):
    if pd.isna(value):
        return False

    text = str(value).strip().casefold()

    if not EXACT_SPECIES_PATTERN.fullmatch(text):
        return False

    if " sp." in text or " sp " in text:
        return False

    return True


events["taxon_exact_species"] = (
    events["species_scientific"]
    .map(is_exact_species_name)
    .astype(bool)
)

identified_labels = (
    events.loc[
        events["identified"],
        "species_scientific",
    ]
    .dropna()
    .drop_duplicates()
)

rejected_labels = sorted(
    label
    for label in identified_labels
    if not is_exact_species_name(label)
)

print("Etiquetas identificadas únicas:", len(identified_labels))
print("Etiquetas rechazadas como especie exacta:", len(rejected_labels))

if rejected_labels:
    rejected_summary = (
        events.loc[
            events["species_scientific"].isin(rejected_labels)
        ]
        .groupby("species_scientific")
        .agg(
            eventos=("event_id", "size"),
            grabaciones=("recording_uid", "nunique"),
            datasets=("dataset", "nunique"),
        )
        .sort_values("eventos", ascending=False)
    )

    display(rejected_summary)

# 3. Auditoría de soporte por especie

El soporte se calcula usando únicamente eventos:

```text
use_detector == True
AND taxon_exact_species == True
```

Por tanto, los mismos eventos utilizados para decidir \(K\) son los que luego podrán entrar al clasificador.

In [ ]:
valid_species_events = events.loc[
    events["use_detector"]
    & events["taxon_exact_species"]
].copy()

print("Eventos válidos con especie exacta:", len(valid_species_events))
print(
    "Especies exactas únicas:",
    valid_species_events["species_scientific"].nunique(),
)

base_support = (
    valid_species_events
    .groupby(
        "species_scientific",
        as_index=False,
    )
    .agg(
        n_events=("event_id", "size"),
        n_recordings=("recording_uid", "nunique"),
        n_datasets=("dataset", "nunique"),
    )
)

event_pivot = (
    valid_species_events
    .pivot_table(
        index="species_scientific",
        columns="dataset",
        values="event_id",
        aggfunc="size",
        fill_value=0,
    )
)

recording_pivot = (
    valid_species_events
    .pivot_table(
        index="species_scientific",
        columns="dataset",
        values="recording_uid",
        aggfunc=pd.Series.nunique,
        fill_value=0,
    )
)

for dataset in ["PteroSet", "WABAD-CO", "Col-CR", "Peru"]:
    if dataset not in event_pivot.columns:
        event_pivot[dataset] = 0

    if dataset not in recording_pivot.columns:
        recording_pivot[dataset] = 0

event_pivot = event_pivot[
    ["PteroSet", "WABAD-CO", "Col-CR", "Peru"]
].copy()

recording_pivot = recording_pivot[
    ["PteroSet", "WABAD-CO", "Col-CR", "Peru"]
].copy()

event_pivot.columns = [
    "events_pteroset",
    "events_wabad_co",
    "events_colcr",
    "events_peru",
]

recording_pivot.columns = [
    "recordings_pteroset",
    "recordings_wabad_co",
    "recordings_colcr",
    "recordings_peru",
]

species_support = (
    base_support
    .set_index("species_scientific")
    .join(event_pivot, how="left")
    .join(recording_pivot, how="left")
    .reset_index()
)

count_columns = [
    "n_events",
    "n_recordings",
    "n_datasets",
    "events_pteroset",
    "events_wabad_co",
    "events_colcr",
    "events_peru",
    "recordings_pteroset",
    "recordings_wabad_co",
    "recordings_colcr",
    "recordings_peru",
]

for column in count_columns:
    species_support[column] = (
        species_support[column]
        .fillna(0)
        .astype(int)
    )

species_support["evidence_pteroset"] = (
    species_support["events_pteroset"] > 0
)

species_support["evidence_wabad_co"] = (
    species_support["events_wabad_co"] > 0
)

species_support["colombian_evidence"] = (
    species_support["evidence_pteroset"]
    | species_support["evidence_wabad_co"]
)

species_support["meets_min_events"] = (
    species_support["n_events"] >= MIN_EVENTS
)

species_support["meets_min_recordings"] = (
    species_support["n_recordings"] >= MIN_RECORDINGS
)

species_support["target_candidate"] = (
    species_support["colombian_evidence"]
    & species_support["meets_min_events"]
    & species_support["meets_min_recordings"]
)

species_support = species_support.sort_values(
    [
        "target_candidate",
        "n_events",
        "n_recordings",
        "species_scientific",
    ],
    ascending=[False, False, False, True],
).reset_index(drop=True)

species_support.to_csv(
    SPECIES_AUDIT_PATH,
    index=False,
)

display(
    species_support.loc[
        species_support["target_candidate"]
    ].reset_index(drop=True)
)

## Valor real de K

No se fuerza ningún número esperado. El resultado de esta celda es el conjunto real de especies que cumple los criterios declarados.

In [ ]:
target_candidates = (
    species_support.loc[
        species_support["target_candidate"],
        "species_scientific",
    ]
    .sort_values()
    .tolist()
)

K = len(target_candidates)

print(f"K REAL = {K}")
print()

for i, species in enumerate(target_candidates, start=1):
    print(f"{i:02d}. {species}")

if K == 0:
    raise RuntimeError(
        "Ninguna especie cumple los criterios. "
        "Revisa la metadata o los umbrales."
    )

# 4. Construcción de `target_species.csv`

Los códigos de clase numéricos son deterministas:

```text
0       -> OTHER_SPECIES
1..K    -> targets ordenados alfabéticamente
```

El nombre científico es la identidad taxonómica canónica. Se conserva también `species_norm` para facilitar compatibilidad con notebooks previos.

In [ ]:
target_species = (
    species_support.loc[
        species_support["target_candidate"]
    ]
    .copy()
    .sort_values("species_scientific")
    .reset_index(drop=True)
)

target_species.insert(
    1,
    "species_norm",
    target_species["species_scientific"],
)

target_species.insert(
    2,
    "class_label",
    target_species["species_scientific"],
)

target_species.insert(
    3,
    "class_code",
    np.arange(1, len(target_species) + 1, dtype=int),
)

target_species = target_species[
    [
        "species_scientific",
        "species_norm",
        "class_label",
        "class_code",
        "n_events",
        "n_recordings",
        "n_datasets",
        "events_pteroset",
        "events_wabad_co",
        "events_colcr",
        "events_peru",
        "recordings_pteroset",
        "recordings_wabad_co",
        "recordings_colcr",
        "recordings_peru",
        "evidence_pteroset",
        "evidence_wabad_co",
        "colombian_evidence",
        "meets_min_events",
        "meets_min_recordings",
    ]
]

display(target_species)

# 5. Conjunto de especies elegibles para `OTHER_SPECIES`

Una especie no target puede formar parte de `OTHER_SPECIES` únicamente si tiene evidencia en PteroSet o WABAD-CO.

Después de establecer esa elegibilidad, también pueden utilizarse sus eventos válidos procedentes de Col-CR o Perú.

In [ ]:
target_set = set(
    target_species["species_scientific"]
)

colombian_species_set = set(
    species_support.loc[
        species_support["colombian_evidence"],
        "species_scientific",
    ]
)

other_species_set = (
    colombian_species_set - target_set
)

print("Especies target:", len(target_set))
print(
    "Especies exactas con evidencia colombiana:",
    len(colombian_species_set),
)
print(
    "Especies elegibles para OTHER_SPECIES:",
    len(other_species_set),
)

# 6. Construcción de `master_manifest.csv`

`master_manifest.csv` conserva todos los eventos fuente, incluyendo:

- eventos sin identificar;
- geometrías inválidas;
- especies identificadas no utilizadas por el clasificador.

Las decisiones de modelado quedan expresadas mediante:

- `is_target`
- `colombian_evidence`
- `class_label`
- `class_code`
- `use_detector`
- `use_classifier`

In [ ]:
master = events.copy()

master["is_target"] = (
    master["species_scientific"].isin(target_set)
)

master["colombian_evidence"] = (
    master["species_scientific"].isin(
        colombian_species_set
    )
)

master["use_classifier"] = (
    master["use_detector"]
    & master["taxon_exact_species"]
    & (
        master["is_target"]
        | master["species_scientific"].isin(
            other_species_set
        )
    )
)

master["class_label"] = pd.Series(
    pd.NA,
    index=master.index,
    dtype="string",
)

target_mask = (
    master["use_classifier"]
    & master["is_target"]
)

other_mask = (
    master["use_classifier"]
    & ~master["is_target"]
)

master.loc[
    target_mask,
    "class_label",
] = master.loc[
    target_mask,
    "species_scientific",
]

master.loc[
    other_mask,
    "class_label",
] = OTHER_LABEL

code_map = dict(
    zip(
        target_species["species_scientific"],
        target_species["class_code"],
    )
)

master["class_code"] = pd.Series(
    pd.NA,
    index=master.index,
    dtype="Int64",
)

master.loc[
    target_mask,
    "class_code",
] = (
    master.loc[
        target_mask,
        "species_scientific",
    ]
    .map(code_map)
    .astype("Int64")
)

master.loc[
    other_mask,
    "class_code",
] = OTHER_CLASS_CODE

recording_lookup = (
    recordings_base[
        [
            "recording_uid",
            "normalized_path",
            "duration_sec",
            "status",
        ]
    ]
    .rename(columns={
        "duration_sec": "recording_duration_sec",
        "status": "recording_status",
    })
)

master = master.merge(
    recording_lookup,
    on="recording_uid",
    how="left",
    validate="many_to_one",
)

if master["normalized_path"].isna().any():
    raise AssertionError(
        "Hay eventos sin ruta de audio normalizado."
    )

MASTER_COLUMNS = [
    "event_id",
    "dataset",
    "recording",
    "recording_key",
    "recording_uid",
    "normalized_path",
    "recording_duration_sec",
    "recording_status",
    "source_group",
    "source_archive",
    "country",
    "species_source_code",
    "species_raw",
    "species_scientific",
    "identified",
    "taxon_exact_species",
    "colombian_evidence",
    "is_target",
    "class_label",
    "class_code",
    "t_min",
    "t_max",
    "f_min",
    "f_max",
    "f_min_original",
    "f_max_original",
    "duration_event_sec",
    "bandwidth_hz",
    "use_detector",
    "use_classifier",
    "audio_available",
]

master = master[
    MASTER_COLUMNS
].copy()

print("Master rows:", len(master))
print("Detector rows:", int(master["use_detector"].sum()))
print("Classifier rows:", int(master["use_classifier"].sum()))

# 7. Construcción de `detector_manifest.csv`

El detector utiliza **todos** los eventos con geometría válida, independientemente de que exista identificación taxonómica.

In [ ]:
detector_manifest = (
    master.loc[
        master["use_detector"]
    ]
    [
        [
            "event_id",
            "dataset",
            "recording",
            "recording_uid",
            "normalized_path",
            "country",
            "species_scientific",
            "identified",
            "t_min",
            "t_max",
            "f_min",
            "f_max",
            "duration_event_sec",
            "bandwidth_hz",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

detector_manifest["detector_label"] = "BIRD_EVENT"

print("Eventos del detector:", len(detector_manifest))

display(
    detector_manifest.groupby("dataset").size().rename("eventos")
)

# 8. Construcción de `classifier_manifest.csv`

El clasificador contiene únicamente:

- eventos con geometría válida;
- taxón exacto;
- targets;
- o especies elegibles para `OTHER_SPECIES`.

No contiene eventos no identificados ni etiquetas taxonómicamente incompletas.

In [ ]:
classifier_manifest = (
    master.loc[
        master["use_classifier"]
    ]
    [
        [
            "event_id",
            "dataset",
            "recording",
            "recording_uid",
            "normalized_path",
            "country",
            "species_scientific",
            "is_target",
            "class_label",
            "class_code",
            "t_min",
            "t_max",
            "f_min",
            "f_max",
            "duration_event_sec",
            "bandwidth_hz",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

classifier_manifest["class_code"] = (
    classifier_manifest["class_code"].astype(int)
)

print("Eventos del clasificador:", len(classifier_manifest))
print(
    "Eventos target:",
    int(classifier_manifest["is_target"].sum()),
)
print(
    "Eventos OTHER_SPECIES:",
    int((~classifier_manifest["is_target"]).sum()),
)
print(
    "Clases del clasificador:",
    classifier_manifest["class_label"].nunique(),
)

class_distribution = (
    classifier_manifest
    .groupby(
        ["class_code", "class_label"],
        as_index=False,
    )
    .agg(
        events=("event_id", "size"),
        recordings=("recording_uid", "nunique"),
        datasets=("dataset", "nunique"),
    )
    .sort_values("class_code")
    .reset_index(drop=True)
)

class_distribution.to_csv(
    CLASS_DISTRIBUTION_PATH,
    index=False,
)

display(class_distribution)

# 9. Construcción de `recordings_manifest.csv`

Se conservan los 1017 audios físicos.

`target_species_present` se calcula **exclusivamente con eventos target válidos para el clasificador**. Por tanto, el número de grabaciones que soporta una especie debe coincidir exactamente con el usado para seleccionarla.

In [ ]:
target_events = classifier_manifest.loc[
    classifier_manifest["is_target"]
].copy()

targets_by_recording = (
    target_events
    .groupby("recording_uid")["species_scientific"]
    .agg(
        lambda values: ";".join(
            sorted(set(values))
        )
    )
)

target_count_by_recording = (
    target_events
    .groupby("recording_uid")["species_scientific"]
    .nunique()
)

classifier_event_count_by_recording = (
    classifier_manifest
    .groupby("recording_uid")
    .size()
)

detector_event_count_by_recording = (
    detector_manifest
    .groupby("recording_uid")
    .size()
)

recordings = recordings_base.copy()

recordings["target_species_present"] = (
    recordings["recording_uid"]
    .map(targets_by_recording)
    .fillna("")
)

recordings["n_target_species_present"] = (
    recordings["recording_uid"]
    .map(target_count_by_recording)
    .fillna(0)
    .astype(int)
)

recordings["classifier_event_count"] = (
    recordings["recording_uid"]
    .map(classifier_event_count_by_recording)
    .fillna(0)
    .astype(int)
)

recordings["detector_event_count"] = (
    recordings["recording_uid"]
    .map(detector_event_count_by_recording)
    .fillna(0)
    .astype(int)
)

recordings = recordings.sort_values(
    [
        "dataset",
        "source_group",
        "source_recording",
    ],
    na_position="last",
).reset_index(drop=True)

print("Filas recordings_manifest:", len(recordings))
print(
    "Grabaciones con al menos un target:",
    int((recordings["n_target_species_present"] > 0).sum()),
)

# 10. Validación crítica de soporte por grabaciones

Para cada target comprobamos:

```text
n_recordings usado para seleccionar K
==
n_recordings observado en classifier_manifest
==
n_recordings derivado de recordings_manifest
```

Si una especie fue seleccionada con 23 grabaciones y luego aparece con 1, el notebook falla.

In [ ]:
support_validation_rows = []

for _, row in target_species.iterrows():
    species = row["species_scientific"]

    expected_events = int(row["n_events"])
    expected_recordings = int(row["n_recordings"])

    classifier_species = (
        classifier_manifest.loc[
            classifier_manifest["species_scientific"].eq(species)
            & classifier_manifest["is_target"]
        ]
    )

    classifier_events = len(classifier_species)
    classifier_recordings = (
        classifier_species["recording_uid"].nunique()
    )

    recordings_manifest_recordings = int(
        recordings["target_species_present"]
        .str.split(";")
        .map(
            lambda labels: (
                species in labels
                if isinstance(labels, list)
                else False
            )
        )
        .sum()
    )

    ok = (
        expected_events == classifier_events
        and expected_recordings == classifier_recordings
        and expected_recordings == recordings_manifest_recordings
    )

    support_validation_rows.append({
        "species_scientific": species,
        "support_events": expected_events,
        "classifier_events": classifier_events,
        "support_recordings": expected_recordings,
        "classifier_recordings": classifier_recordings,
        "recordings_manifest_recordings": recordings_manifest_recordings,
        "ok": ok,
    })

support_validation = pd.DataFrame(
    support_validation_rows
)

display(support_validation)

if not support_validation["ok"].all():
    raise AssertionError(
        "Inconsistencia entre la selección de targets y los manifests finales."
    )

print("Soporte de todas las especies target consistente.")

# 11. Resumen final por dataset

In [ ]:
physical_summary = (
    recordings.groupby("dataset")
    .agg(
        physical_recordings=("recording_uid", "size"),
        annotated_recordings=("has_annotations", "sum"),
        hours=("duration_sec", lambda x: x.sum() / 3600),
    )
)

event_summary = (
    master.groupby("dataset")
    .agg(
        source_events=("event_id", "size"),
        identified_events=("identified", "sum"),
        detector_events=("use_detector", "sum"),
        classifier_events=("use_classifier", "sum"),
        target_events=("is_target", lambda x: 0),
    )
)

# target_events debe considerar solo eventos target que efectivamente usa el clasificador.
target_counts = (
    master.loc[
        master["use_classifier"]
        & master["is_target"]
    ]
    .groupby("dataset")
    .size()
)

other_counts = (
    master.loc[
        master["use_classifier"]
        & ~master["is_target"]
    ]
    .groupby("dataset")
    .size()
)

event_summary["target_events"] = (
    event_summary.index.to_series()
    .map(target_counts)
    .fillna(0)
    .astype(int)
)

event_summary["other_species_events"] = (
    event_summary.index.to_series()
    .map(other_counts)
    .fillna(0)
    .astype(int)
)

dataset_summary = (
    physical_summary
    .join(event_summary, how="outer")
    .reset_index()
)

dataset_summary["audio_only_recordings"] = (
    dataset_summary["physical_recordings"]
    - dataset_summary["annotated_recordings"]
)

integer_columns = [
    "physical_recordings",
    "annotated_recordings",
    "audio_only_recordings",
    "source_events",
    "identified_events",
    "detector_events",
    "classifier_events",
    "target_events",
    "other_species_events",
]

for column in integer_columns:
    dataset_summary[column] = (
        dataset_summary[column]
        .fillna(0)
        .astype(int)
    )

display(dataset_summary)

# 12. Validaciones fuertes finales

Estas validaciones deben pasar antes de guardar y antes de subir el corpus a Kaggle.

In [ ]:
checks = []

def add_check(name, condition, observed, expected):
    checks.append({
        "check": name,
        "ok": bool(condition),
        "observed": observed,
        "expected": expected,
    })


add_check(
    "master_mantiene_todos_los_eventos",
    len(master) == len(events),
    len(master),
    len(events),
)

add_check(
    "event_id_unicos_master",
    master["event_id"].is_unique,
    int(master["event_id"].nunique()),
    len(master),
)

add_check(
    "recordings_mantiene_1017_audios",
    len(recordings) == 1017,
    len(recordings),
    1017,
)

add_check(
    "recording_uid_unicos",
    recordings["recording_uid"].is_unique,
    int(recordings["recording_uid"].nunique()),
    1017,
)

add_check(
    "detector_usa_solo_geometria_valida",
    detector_manifest["event_id"].isin(
        master.loc[
            master["use_detector"],
            "event_id",
        ]
    ).all()
    and len(detector_manifest) == int(master["use_detector"].sum()),
    len(detector_manifest),
    int(master["use_detector"].sum()),
)

add_check(
    "detector_esperado_48430",
    len(detector_manifest) == 48430,
    len(detector_manifest),
    48430,
)

add_check(
    "classifier_solo_use_classifier",
    classifier_manifest["event_id"].isin(
        master.loc[
            master["use_classifier"],
            "event_id",
        ]
    ).all()
    and len(classifier_manifest) == int(master["use_classifier"].sum()),
    len(classifier_manifest),
    int(master["use_classifier"].sum()),
)

add_check(
    "classifier_sin_nulos_de_clase",
    classifier_manifest["class_label"].notna().all()
    and classifier_manifest["class_code"].notna().all(),
    int(classifier_manifest["class_label"].isna().sum()),
    0,
)

add_check(
    "classifier_solo_taxones_exactos",
    master.loc[
        master["use_classifier"],
        "taxon_exact_species",
    ].all(),
    bool(
        master.loc[
            master["use_classifier"],
            "taxon_exact_species",
        ].all()
    ),
    True,
)

add_check(
    "classifier_sin_no_identificados",
    master.loc[
        master["use_classifier"],
        "identified",
    ].all(),
    bool(
        master.loc[
            master["use_classifier"],
            "identified",
        ].all()
    ),
    True,
)

add_check(
    "numero_clases_classifier",
    classifier_manifest["class_label"].nunique() == K + 1,
    int(classifier_manifest["class_label"].nunique()),
    K + 1,
)

add_check(
    "target_species_unicos",
    target_species["species_scientific"].is_unique,
    int(target_species["species_scientific"].nunique()),
    K,
)

add_check(
    "class_code_target_unicos",
    target_species["class_code"].is_unique,
    int(target_species["class_code"].nunique()),
    K,
)

add_check(
    "todos_targets_cumplen_eventos",
    (target_species["n_events"] >= MIN_EVENTS).all(),
    int(target_species["n_events"].min()),
    f">={MIN_EVENTS}",
)

add_check(
    "todos_targets_cumplen_grabaciones",
    (target_species["n_recordings"] >= MIN_RECORDINGS).all(),
    int(target_species["n_recordings"].min()),
    f">={MIN_RECORDINGS}",
)

add_check(
    "todos_targets_tienen_evidencia_colombiana",
    target_species["colombian_evidence"].all(),
    bool(target_species["colombian_evidence"].all()),
    True,
)

add_check(
    "support_consistente_por_target",
    support_validation["ok"].all(),
    bool(support_validation["ok"].all()),
    True,
)

other_classifier_species = set(
    classifier_manifest.loc[
        ~classifier_manifest["is_target"],
        "species_scientific",
    ]
)

add_check(
    "other_species_tienen_evidencia_colombiana",
    other_classifier_species <= colombian_species_set,
    len(other_classifier_species - colombian_species_set),
    0,
)

add_check(
    "other_species_no_contiene_targets",
    other_classifier_species.isdisjoint(target_set),
    len(other_classifier_species & target_set),
    0,
)

audio_only_uids = set(
    recordings.loc[
        ~recordings["has_annotations"],
        "recording_uid",
    ]
)

referenced_uids = set(master["recording_uid"])

add_check(
    "audio_only_no_usado_como_background",
    audio_only_uids.isdisjoint(referenced_uids),
    len(audio_only_uids & referenced_uids),
    0,
)

integrity_report = pd.DataFrame(checks)

display(integrity_report)

if not integrity_report["ok"].all():
    failed = integrity_report.loc[
        ~integrity_report["ok"]
    ]

    display(failed)

    raise AssertionError(
        "Fallaron validaciones finales. "
        "No se deben publicar estos manifests."
    )

print("Todas las validaciones finales pasaron.")

In [ ]:
current_target_events = int(
    classifier_manifest["is_target"].sum()
)

current_other_events = int(
    (~classifier_manifest["is_target"]).sum()
)

historical = pd.DataFrame([
    {
        "metric": "K",
        "historical": 30,
        "current": K,
    },
    {
        "metric": "classifier_events",
        "historical": 25589,
        "current": len(classifier_manifest),
    },
    {
        "metric": "target_events",
        "historical": 14405,
        "current": current_target_events,
    },
    {
        "metric": "OTHER_SPECIES_events",
        "historical": 11184,
        "current": current_other_events,
    },
])

historical["matches"] = (
    historical["historical"]
    == historical["current"]
)

display(historical)

# 14. Configuración reproducible

Se guarda la política de modelado junto con los manifests para que la selección de clases sea auditable.

In [ ]:
modeling_config = {
    "min_events": MIN_EVENTS,
    "min_recordings": MIN_RECORDINGS,
    "colombian_eligibility_datasets": sorted(
        COLOMBIAN_ELIGIBILITY_DATASETS
    ),
    "target_selection_uses_only_valid_geometry": True,
    "target_selection_requires_exact_species": True,
    "other_species_requires_colombian_evidence": True,
    "other_species_class_code": OTHER_CLASS_CODE,
    "other_species_label": OTHER_LABEL,
    "class_code_policy": (
        "0=OTHER_SPECIES; target species sorted alphabetically use 1..K"
    ),
    "k": K,
    "target_species": target_species[
        "species_scientific"
    ].tolist(),
}

print(json.dumps(
    modeling_config,
    ensure_ascii=False,
    indent=2,
))

# 15. Guardado final

Los archivos se escriben **solo después** de que todas las validaciones hayan pasado.

In [ ]:
master.to_csv(
    MASTER_PATH,
    index=False,
)

recordings.to_csv(
    RECORDINGS_PATH,
    index=False,
)

target_species.to_csv(
    TARGET_SPECIES_PATH,
    index=False,
)

detector_manifest.to_csv(
    DETECTOR_PATH,
    index=False,
)

classifier_manifest.to_csv(
    CLASSIFIER_PATH,
    index=False,
)

dataset_summary.to_csv(
    DATASET_SUMMARY_PATH,
    index=False,
)

integrity_report.to_csv(
    INTEGRITY_REPORT_PATH,
    index=False,
)

with MODELING_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        modeling_config,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Archivos finales generados:\n")

for path in [
    MASTER_PATH,
    RECORDINGS_PATH,
    TARGET_SPECIES_PATH,
    DETECTOR_PATH,
    CLASSIFIER_PATH,
    DATASET_SUMMARY_PATH,
    MODELING_CONFIG_PATH,
    SPECIES_AUDIT_PATH,
    CLASS_DISTRIBUTION_PATH,
    INTEGRITY_REPORT_PATH,
]:
    print(path)